# D7 Stage 2d Acceptance Notebook

Independent human-in-the-loop acceptance gate for D7 Stage 2d's four committed pre-fire artifacts. This notebook re-derives claims from raw sources and call records; it does not import any Stage 2d derivation/build scripts, make LLM calls, run backtests, or mutate reviewed artifacts.


## Section A - Setup and artifact loading

This section pins paths, expected hashes, and helper functions, then loads the four Stage 2d artifacts. The next code cell also installs a guard against circular verification by asserting that no forbidden Stage 2d builder module is imported.


In [1]:
from __future__ import annotations

import hashlib
import json
import re
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / '.git').exists() and (candidate / 'docs').exists() and (candidate / 'raw_payloads').exists():
            return candidate
    raise AssertionError(f'Could not locate repo root from cwd={start}')


REPO_ROOT = find_repo_root(Path.cwd())
print(f'Resolved repo root: {REPO_ROOT}')

NOTEBOOK_PATH = REPO_ROOT / 'docs/test_notebooks/D7 Stage 2d acceptance notebook.ipynb'
SOURCE_BATCH_UUID = '5cf76668-47d1-48d7-bd90-db06d31982ed'
BATCH_DIR = REPO_ROOT / f'raw_payloads/batch_{SOURCE_BATCH_UUID}'
SUMMARY_PATH = BATCH_DIR / 'stage2d_summary.json'

ARTIFACT_PATHS = {
    'label_universe_analysis_json': REPO_ROOT / 'docs/d7_stage2d/label_universe_analysis.json',
    'replay_candidates_json': REPO_ROOT / 'docs/d7_stage2d/replay_candidates.json',
    'deep_dive_candidates_json': REPO_ROOT / 'docs/d7_stage2d/deep_dive_candidates.json',
    'test_retest_baselines_json': REPO_ROOT / 'docs/d7_stage2d/test_retest_baselines.json',
    'derive_label_universes_py': REPO_ROOT / 'scripts/derive_d7_stage2d_label_universes.py',
    'build_replay_candidates_py': REPO_ROOT / 'scripts/build_d7_stage2d_replay_candidates.py',
    'build_deep_dive_py': REPO_ROOT / 'scripts/build_d7_stage2d_deep_dive.py',
    'build_baselines_py': REPO_ROOT / 'scripts/build_d7_stage2d_baselines.py',
    'scope_lock_v2_md': REPO_ROOT / 'docs/d7_stage2d/D7_STAGE2D_SCOPE_LOCK_v2.md',
}

EXPECTED_SHA256 = {
    'label_universe_analysis_json': 'ecd52a9e7656d31d13a8e62ec11a7345f8966343172fbfc3f95add2762b3e4a0',
    'replay_candidates_json': '05706642cbeb5014d5072c172b343b01ee56d0eb8ee45afb2f3ce6e56665907e',
    'deep_dive_candidates_json': '6cdcd1d22d785d6d58317f61cd62fe8b3340bd22f1ec7c7dcb90e5d2b2da7ce7',
    'test_retest_baselines_json': '5840b90a57206b01e8109ea73b549cf50089964f5cb1f9f7e83b963569adac2f',
    'derive_label_universes_py': '1b595fb64ffc72f114189391ea2391d3ce67c53e8fc64c1d8b285ca8e0755ed0',
    'build_replay_candidates_py': 'd856294bfe6031307432c0e59652ecc321d6d6b37d27c4da32c5bb8aafa2e084',
    'build_deep_dive_py': '93cc0b2359d53e129092823353b6daa56beb0367f15b3b2b1f53d36cbd930e1c',
    'build_baselines_py': '87c9431eec227a3d8ab5e7e686be54d1103cf0ed86c09dc22e9fc67ba09b64a3',
    'scope_lock_v2_md': 'b119067ca4ed3dd3ce8a6ee5c29d62f319187160197e5bf0ab1399beece68f7a',
}

TASK_COMMITS = {
    '2A': '1ce179f8317a9004fc9ba4af290a93639386fd9a',
    '2B': '2771bef9b79db00ac421728b5f69fd908038ce9d',
    '2C': '3d07c881c46d830c92e22aebb435c695c8451670',
    '2D': '50ed3cb9146ee51273fe7baf8062a4a4abe1f251',
}
SCOPE_LOCK_COMMIT = '4303d8de2882362ec55c8c581519331c5f6404c6'

AGREEMENT_LABEL = 'agreement_expected'
DIVERGENCE_LABEL = 'divergence_expected'
NEUTRAL_LABEL = 'neutral'
LABEL_ORDER = [AGREEMENT_LABEL, DIVERGENCE_LABEL, NEUTRAL_LABEL]
STAGE2C_OVERLAP_POSITIONS = [17, 22, 27, 32, 62, 72, 73, 74, 77, 83, 97, 102, 107, 112, 117, 138, 143, 147, 152, 162]
STAGE2C_OVERLAP_SET = set(STAGE2C_OVERLAP_POSITIONS)
TIER1_POSITIONS = [17, 73, 74, 97, 138]
TIER2_POSITIONS = [22, 27, 32, 62, 72, 77, 83, 102, 107, 112, 117, 143, 147, 152, 162]
FRESH_9 = {122, 127, 128, 129, 132, 172, 178, 182, 187}
EXPECTED_FRESH7_RSI_ABSENT_VOL = {3, 43, 68, 128, 173, 188, 198}
EXPECTED_UNIVERSE_A_COUNTS = {AGREEMENT_LABEL: 11, DIVERGENCE_LABEL: 3, NEUTRAL_LABEL: 15}
EXPECTED_UNIVERSE_B_COUNTS = {AGREEMENT_LABEL: 66, DIVERGENCE_LABEL: 5, NEUTRAL_LABEL: 128}
EXPECTED_STAGE2B_REASONING_LENGTHS = [1237, 1116, 1050, 1106, 1164]
EXPECTED_STAGE2B_SVR = {17: 0.85, 73: 0.85, 74: 0.65, 97: 0.95, 138: 0.15}
EXPECTED_STAGE2C_SVR = {
    17: 0.85, 22: 0.85, 27: 0.85, 32: 0.90, 62: 0.95,
    72: 0.75, 73: 0.95, 74: 0.75, 77: 0.95, 83: 0.75,
    97: 0.95, 102: 0.95, 107: 0.95, 112: 0.85, 117: 0.85,
    138: 0.30, 143: 0.15, 147: 0.95, 152: 0.95, 162: 0.90,
}

FORBIDDEN_MODULES = {
    'scripts.derive_d7_stage2d_label_universes',
    'scripts.build_d7_stage2d_replay_candidates',
    'scripts.build_d7_stage2d_deep_dive',
    'scripts.build_d7_stage2d_baselines',
}

checks = []


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_json(path: Path):
    with path.open('r', encoding='utf-8') as fh:
        return json.load(fh)


def check(section: str, name: str, condition: bool, notes: str = '', *, material: bool = True):
    status = 'PASS' if condition else 'FAIL'
    checks.append({'section': section, 'check': name, 'passed': bool(condition), 'material': material, 'notes': notes})
    print(f'[{section}] {status}: {name} -- {notes}')
    assert condition, f'{section} {name}: {notes}'


def check_equal(section: str, name: str, observed, expected, *, material: bool = True):
    check(section, name, observed == expected, f'observed={observed!r}; expected={expected!r}', material=material)


def git_output(args: list[str]) -> str:
    return subprocess.check_output(['git', *args], cwd=REPO_ROOT, text=True).strip()

loaded_forbidden = sorted(m for m in FORBIDDEN_MODULES if m in sys.modules)
check_equal('A', 'forbidden Stage 2d builder modules not imported', loaded_forbidden, [])
print('Setup complete. Constants pinned and circular-import guard passed.')


Resolved repo root: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline
[A] PASS: forbidden Stage 2d builder modules not imported -- observed=[]; expected=[]
Setup complete. Constants pinned and circular-import guard passed.


### A1. Load Stage 2d artifacts

The next cell loads the four committed Stage 2d JSON artifacts and displays their top-level shapes. This is only setup; substantive checks are re-derived later from raw batch and call records.


In [2]:
label_universe = load_json(ARTIFACT_PATHS['label_universe_analysis_json'])
replay_candidates = load_json(ARTIFACT_PATHS['replay_candidates_json'])
deep_dive_candidates = load_json(ARTIFACT_PATHS['deep_dive_candidates_json'])
test_retest_baselines = load_json(ARTIFACT_PATHS['test_retest_baselines_json'])
stage2c_replay_candidates = load_json(REPO_ROOT / 'docs/d7_stage2c/replay_candidates.json')
stage2b_replay_candidates = load_json(REPO_ROOT / 'docs/d7_stage2b/replay_candidates.json')

shape_rows = [
    {'artifact': 'label_universe_analysis.json', 'top_level_keys': len(label_universe), 'primary_count': label_universe['source_n']},
    {'artifact': 'replay_candidates.json', 'top_level_keys': len(replay_candidates), 'primary_count': len(replay_candidates['candidates'])},
    {'artifact': 'deep_dive_candidates.json', 'top_level_keys': len(deep_dive_candidates), 'primary_count': len(deep_dive_candidates['candidates'])},
    {'artifact': 'test_retest_baselines.json', 'top_level_keys': len(test_retest_baselines), 'primary_count': len(test_retest_baselines['baselines'])},
]
shape_df = pd.DataFrame(shape_rows)
display(shape_df)
check_equal('A', 'four Stage 2d artifacts loaded', len(shape_df), 4)


,artifact,top_level_keys,primary_count
0,label_universe_analysis.json,11,200
1,replay_candidates.json,8,200
2,deep_dive_candidates.json,11,20
3,test_retest_baselines.json,10,20


[A] PASS: four Stage 2d artifacts loaded -- observed=4; expected=4


### A2. SHA256 and commit anchors

This cell hashes the four JSON artifacts, four Stage 2d scripts, and the scope lock. It also confirms the scope-lock commit exists and is the latest commit that touched the scope-lock file.


In [3]:
sha_rows = []
for key, path in ARTIFACT_PATHS.items():
    observed = sha256_file(path)
    expected = EXPECTED_SHA256[key]
    sha_rows.append({'key': key, 'path': str(path.relative_to(REPO_ROOT)), 'observed_sha256': observed, 'expected_sha256': expected, 'match': observed == expected})
sha_df = pd.DataFrame(sha_rows)
display(sha_df)
check_equal('A', 'all pinned Stage 2d and scope-lock SHAs match', int(sha_df['match'].sum()), len(sha_df))

resolved_scope = git_output(['rev-parse', SCOPE_LOCK_COMMIT])
scope_type = git_output(['cat-file', '-t', SCOPE_LOCK_COMMIT])
latest_scope_commit = git_output(['log', '-1', '--format=%H', '--', 'docs/d7_stage2d/D7_STAGE2D_SCOPE_LOCK_v2.md'])
check_equal('A', 'scope lock commit resolves', resolved_scope, SCOPE_LOCK_COMMIT)
check_equal('A', 'scope lock object is commit', scope_type, 'commit')
check_equal('A', 'scope lock commit is latest touching scope lock file', latest_scope_commit, SCOPE_LOCK_COMMIT)

notebook_self_sha = sha256_file(NOTEBOOK_PATH) if NOTEBOOK_PATH.exists() else None
print(f'Notebook self SHA at execution time: {notebook_self_sha}')


,key,path,observed_sha256,expected_sha256,match
0,label_universe_analysis_json,docs/d7_stage2d/label_universe_analysis.json,ecd52a9e7656d31d13a8e62ec11a7345f8966343172fbf...,ecd52a9e7656d31d13a8e62ec11a7345f8966343172fbf...,True
1,replay_candidates_json,docs/d7_stage2d/replay_candidates.json,05706642cbeb5014d5072c172b343b01ee56d0eb8ee45a...,05706642cbeb5014d5072c172b343b01ee56d0eb8ee45a...,True
2,deep_dive_candidates_json,docs/d7_stage2d/deep_dive_candidates.json,6cdcd1d22d785d6d58317f61cd62fe8b3340bd22f1ec7c...,6cdcd1d22d785d6d58317f61cd62fe8b3340bd22f1ec7c...,True
3,test_retest_baselines_json,docs/d7_stage2d/test_retest_baselines.json,5840b90a57206b01e8109ea73b549cf50089964f5cb1f9...,5840b90a57206b01e8109ea73b549cf50089964f5cb1f9...,True
4,derive_label_universes_py,scripts/derive_d7_stage2d_label_universes.py,1b595fb64ffc72f114189391ea2391d3ce67c53e8fc64c...,1b595fb64ffc72f114189391ea2391d3ce67c53e8fc64c...,True
5,build_replay_candidates_py,scripts/build_d7_stage2d_replay_candidates.py,d856294bfe6031307432c0e59652ecc321d6d6b37d27c4...,d856294bfe6031307432c0e59652ecc321d6d6b37d27c4...,True
6,build_deep_dive_py,scripts/build_d7_stage2d_deep_dive.py,93cc0b2359d53e129092823353b6daa56beb0367f15b3b...,93cc0b2359d53e129092823353b6daa56beb0367f15b3b...,True
7,build_baselines_py,scripts/build_d7_stage2d_baselines.py,87c9431eec227a3d8ab5e7e686be54d1103cf0ed86c09d...,87c9431eec227a3d8ab5e7e686be54d1103cf0ed86c09d...,True
8,scope_lock_v2_md,docs/d7_stage2d/D7_STAGE2D_SCOPE_LOCK_v2.md,b119067ca4ed3dd3ce8a6ee5c29d62f319187160197e5b...,b119067ca4ed3dd3ce8a6ee5c29d62f319187160197e5b...,True


[A] PASS: all pinned Stage 2d and scope-lock SHAs match -- observed=9; expected=9
[A] PASS: scope lock commit resolves -- observed='4303d8de2882362ec55c8c581519331c5f6404c6'; expected='4303d8de2882362ec55c8c581519331c5f6404c6'
[A] PASS: scope lock object is commit -- observed='commit'; expected='commit'
[A] PASS: scope lock commit is latest touching scope lock file -- observed='4303d8de2882362ec55c8c581519331c5f6404c6'; expected='4303d8de2882362ec55c8c581519331c5f6404c6'
Notebook self SHA at execution time: 84e439f2caeeafaedd2951a7acb831401f38e101e24f762e8d6370f992e7f8cc


## Section B - Source batch foundation

This section verifies the raw D6 Stage 2d batch foundation before any derived artifact is trusted.


In [4]:
summary = load_json(SUMMARY_PATH)
summary_sha = sha256_file(SUMMARY_PATH)
calls = list(summary['calls'])
by_position = {c['position']: c for c in calls}
pending_positions = sorted(c['position'] for c in calls if c.get('lifecycle_state') == 'pending_backtest')

foundation_df = pd.DataFrame([
    {'metric': 'stage2d_summary_sha256', 'value': summary_sha},
    {'metric': 'source_call_count', 'value': len(calls)},
    {'metric': 'pending_backtest_count', 'value': len(pending_positions)},
    {'metric': 'position_116_lifecycle', 'value': by_position[116].get('lifecycle_state')},
    {'metric': 'position_116_valid_status', 'value': by_position[116].get('valid_status')},
])
display(foundation_df)
check_equal('B', 'raw summary has exactly 200 calls', len(calls), 200)
check_equal('B', 'raw positions are exactly 1..200', sorted(by_position), list(range(1, 201)))
check_equal('B', 'position 116 lifecycle state', by_position[116].get('lifecycle_state'), 'rejected_complexity')
check_equal('B', 'position 116 valid status', by_position[116].get('valid_status'), 'invalid_schema')
check_equal('B', 'exactly 199 pending_backtest positions', len(pending_positions), 199)


,metric,value
0,stage2d_summary_sha256,3646269e868d390688d78bdb90e63df372f5159f82d0b3...
1,source_call_count,200
2,pending_backtest_count,199
3,position_116_lifecycle,rejected_complexity
4,position_116_valid_status,invalid_schema


[B] PASS: raw summary has exactly 200 calls -- observed=200; expected=200
[B] PASS: raw positions are exactly 1..200 -- observed=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196

## Section C - Universe A and Universe B independent re-derivation

This section rewrites the selector eligibility predicate and relationship-label rule inside the notebook. It then independently rebuilds Universe A and Universe B from the raw batch and compares them against `label_universe_analysis.json`.


In [5]:
MIN_POSITION = 10
MAX_POSITION = 190
STAGE2B_N_FACTORS_RANGE = (3, 7)
THIN_THEMES = {'volume_divergence', 'calendar_effect', 'volatility_regime'}


def find_response_path(position: int) -> Path | None:
    primary = BATCH_DIR / f'attempt_{position:04d}_response.txt'
    if primary.exists():
        return primary
    retries = sorted(BATCH_DIR.glob(f'attempt_{position:04d}_retry_*_response.txt'))
    return retries[-1] if retries else None


def has_cross_operator(response_text: str) -> bool:
    try:
        payload = json.loads(response_text)
    except json.JSONDecodeError:
        return ('crosses_above' in response_text) or ('crosses_below' in response_text)
    if not isinstance(payload, dict):
        return ('crosses_above' in response_text) or ('crosses_below' in response_text)
    for block_key in ('entry', 'exit'):
        block = payload.get(block_key)
        if not isinstance(block, list):
            continue
        for group in block:
            if not isinstance(group, dict):
                continue
            conditions = group.get('conditions', [])
            if not isinstance(conditions, list):
                continue
            for cond in conditions:
                if isinstance(cond, dict) and (cond.get('op') or cond.get('operator')) in {'crosses_above', 'crosses_below'}:
                    return True
    return ('crosses_above' in response_text) or ('crosses_below' in response_text)


def thin_theme_momentum_bleed(call: dict) -> bool:
    return call.get('theme') in THIN_THEMES and len(call.get('default_momentum_factors_used') or []) >= 2


def eligibility_reason(call: dict) -> str:
    pos = call.get('position')
    factors = call.get('factors_used') or []
    if call.get('lifecycle_state') != 'pending_backtest':
        return 'lifecycle_state != pending_backtest'
    if call.get('theme') == 'momentum':
        return 'theme == momentum'
    if not (STAGE2B_N_FACTORS_RANGE[0] <= len(factors) <= STAGE2B_N_FACTORS_RANGE[1]):
        return f'n_factors={len(factors)} outside [3,7]'
    if factors == ['rsi_14']:
        return 'rsi_14 is sole factor'
    if pos is None or not (MIN_POSITION <= pos <= MAX_POSITION):
        return f'position={pos} outside [10,190]'
    if thin_theme_momentum_bleed(call):
        return 'thin_theme_momentum_bleed would fire'
    response_path = find_response_path(pos)
    if response_path is None:
        return f'response file not found for position {pos}'
    try:
        response_text = response_path.read_text(encoding='utf-8')
    except OSError as exc:
        return f'cannot read response: {exc}'
    if not has_cross_operator(response_text):
        return 'no crosses_above/crosses_below operator'
    return 'all criteria met'


def passes_independent_eligibility(call: dict) -> bool:
    return eligibility_reason(call) == 'all criteria met'


def max_overlap(f_current: set[str], f_priors: list[set[str]]) -> int:
    return max((len(f_current & p) for p in f_priors), default=0)


def relationship_label(prior_occurrences: int, factor_set: set[str], priors: list[set[str]]) -> str:
    if prior_occurrences > 0:
        return AGREEMENT_LABEL
    if max_overlap(factor_set, priors) <= 2:
        return DIVERGENCE_LABEL
    return NEUTRAL_LABEL


def sorted_calls_for_scan(source_calls: list[dict]) -> list[dict]:
    return sorted(source_calls, key=lambda c: (c.get('position', float('inf')), c.get('hypothesis_hash') or ''))


def derive_labels(source_calls: list[dict], *, apply_eligibility: bool) -> dict[int, dict]:
    occurrence_counter = Counter()
    seen_factor_sets = set()
    distinct_priors: list[set[str]] = []
    out = {}
    scan_calls = sorted_calls_for_scan(source_calls)
    if apply_eligibility:
        scan_calls = [c for c in scan_calls if passes_independent_eligibility(c)]
    else:
        scan_calls = [c for c in scan_calls if c.get('lifecycle_state') == 'pending_backtest']
    for call in scan_calls:
        factors_tuple = tuple(sorted(call.get('factors_used') or []))
        factor_set = set(factors_tuple)
        prior_occurrences = occurrence_counter[factors_tuple]
        overlap = max_overlap(factor_set, distinct_priors) if factor_set else 0
        label = relationship_label(prior_occurrences, factor_set, distinct_priors)
        out[call['position']] = {
            'position': call['position'],
            'theme': call.get('theme'),
            'factors_used': list(call.get('factors_used') or []),
            'factor_set_size': len(call.get('factors_used') or []),
            'factor_set_prior_occurrences': prior_occurrences,
            'max_overlap_with_priors': overlap,
            'label': label,
        }
        occurrence_counter[factors_tuple] += 1
        if factor_set and factors_tuple not in seen_factor_sets:
            seen_factor_sets.add(factors_tuple)
            distinct_priors.append(factor_set)
    return out

eligibility_breakdown = Counter(eligibility_reason(c) for c in sorted_calls_for_scan(calls))
eligibility_df = pd.DataFrame([{'reason': k, 'count': v} for k, v in sorted(eligibility_breakdown.items())])
display(eligibility_df)
check_equal('C', 'independent Universe A eligibility count is 29', eligibility_breakdown['all criteria met'], 29)


,reason,count
0,all criteria met,29
1,lifecycle_state != pending_backtest,1
2,"n_factors=8 outside [3,7]",1
3,no crosses_above/crosses_below operator,16
4,"position=192 outside [10,190]",1
5,"position=193 outside [10,190]",1
6,"position=194 outside [10,190]",1
7,"position=195 outside [10,190]",1
8,"position=197 outside [10,190]",1
9,"position=198 outside [10,190]",1


[C] PASS: independent Universe A eligibility count is 29 -- observed=29; expected=29


### C3-C5. Re-derived universes and overlap-conflict comparison

The next cell derives Universe A and B, compares them to `label_universe_analysis.json`, and rebuilds the Stage 2c frozen-label conflict table from the Stage 2c selection artifact.


In [6]:
derived_a = derive_labels(calls, apply_eligibility=True)
derived_b = derive_labels(calls, apply_eligibility=False)

def positions_by_label(derived: dict[int, dict]) -> dict[str, list[int]]:
    return {label: sorted(pos for pos, row in derived.items() if row['label'] == label) for label in LABEL_ORDER}

def counts_by_label(derived: dict[int, dict]) -> dict[str, int]:
    return {label: len(positions_by_label(derived)[label]) for label in LABEL_ORDER}

def artifact_positions_by_label(doc: dict, universe_key: str) -> dict[str, list[int]]:
    return {label: sorted(doc[universe_key]['candidate_positions'][label]) for label in LABEL_ORDER}

ua_positions = positions_by_label(derived_a)
ub_positions = positions_by_label(derived_b)
artifact_ua_positions = artifact_positions_by_label(label_universe, 'universe_a')
artifact_ub_positions = artifact_positions_by_label(label_universe, 'universe_b')

universe_summary = pd.DataFrame([
    {'universe': 'A_derived', **counts_by_label(derived_a), 'size': len(derived_a)},
    {'universe': 'A_artifact', **label_universe['universe_a']['counts'], 'size': label_universe['universe_a']['size']},
    {'universe': 'B_derived', **counts_by_label(derived_b), 'size': len(derived_b)},
    {'universe': 'B_artifact', **label_universe['universe_b']['counts'], 'size': label_universe['universe_b']['size']},
])
display(universe_summary)
check_equal('C', 'Universe A derived label counts', counts_by_label(derived_a), EXPECTED_UNIVERSE_A_COUNTS)
check_equal('C', 'Universe A size', len(derived_a), 29)
check_equal('C', 'Universe A position buckets match artifact', ua_positions, artifact_ua_positions)
check_equal('C', 'Universe B derived label counts', counts_by_label(derived_b), EXPECTED_UNIVERSE_B_COUNTS)
check_equal('C', 'Universe B size', len(derived_b), 199)
check_equal('C', 'Universe B position buckets match artifact', ub_positions, artifact_ub_positions)

# Per-position diff table, intentionally displayed even when empty.
diff_rows = []
for universe_name, derived, artifact in [('A', derived_a, artifact_ua_positions), ('B', derived_b, artifact_ub_positions)]:
    artifact_map = {pos: label for label, positions in artifact.items() for pos in positions}
    all_positions = sorted(set(derived) | set(artifact_map))
    for pos in all_positions:
        d_label = derived.get(pos, {}).get('label')
        a_label = artifact_map.get(pos)
        if d_label != a_label:
            diff_rows.append({'universe': universe_name, 'position': pos, 'derived_label': d_label, 'artifact_label': a_label})
diff_df = pd.DataFrame(diff_rows, columns=['universe', 'position', 'derived_label', 'artifact_label'])
display(diff_df)
check_equal('C', 'Universe A/B per-position diffs', len(diff_df), 0)

stage2c_frozen_map = {c['position']: c['d7a_b_relationship_label'] for c in stage2c_replay_candidates['candidates']}
overlap_rows = []
for pos in STAGE2C_OVERLAP_POSITIONS:
    frozen = stage2c_frozen_map[pos]
    ub = derived_b[pos]['label']
    overlap_rows.append({'position': pos, 'stage2c_frozen_label': frozen, 'universe_b_label': ub, 'conflict': frozen != ub})
overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df)
conflict_positions = set(overlap_df.loc[overlap_df['conflict'], 'position'])
check_equal('C', 'Stage 2c overlap conflict positions', conflict_positions, {17, 73, 74})
check_equal('C', 'Stage 2c conflicts are divergence to neutral', overlap_df[overlap_df['conflict']][['stage2c_frozen_label', 'universe_b_label']].drop_duplicates().to_dict('records'), [{'stage2c_frozen_label': DIVERGENCE_LABEL, 'universe_b_label': NEUTRAL_LABEL}])


,universe,agreement_expected,divergence_expected,neutral,size
0,A_derived,11,3,15,29
1,A_artifact,11,3,15,29
2,B_derived,66,5,128,199
3,B_artifact,66,5,128,199


[C] PASS: Universe A derived label counts -- observed={'agreement_expected': 11, 'divergence_expected': 3, 'neutral': 15}; expected={'agreement_expected': 11, 'divergence_expected': 3, 'neutral': 15}
[C] PASS: Universe A size -- observed=29; expected=29
[C] PASS: Universe A position buckets match artifact -- observed={'agreement_expected': [27, 97, 102, 107, 112, 122, 127, 132, 147, 152, 162], 'divergence_expected': [17, 73, 74], 'neutral': [22, 32, 62, 72, 77, 83, 117, 128, 129, 138, 143, 172, 178, 182, 187]}; expected={'agreement_expected': [27, 97, 102, 107, 112, 122, 127, 132, 147, 152, 162], 'divergence_expected': [17, 73, 74], 'neutral': [22, 32, 62, 72, 77, 83, 117, 128, 129, 138, 143, 172, 178, 182, 187]}
[C] PASS: Universe B derived label counts -- observed={'agreement_expected': 66, 'divergence_expected': 5, 'neutral': 128}; expected={'agreement_expected': 66, 'divergence_expected': 5, 'neutral': 128}
[C] PASS: Universe B size -- observed=199; expected=199
[C] PASS: Universe 

,universe,position,derived_label,artifact_label


[C] PASS: Universe A/B per-position diffs -- observed=0; expected=0


,position,stage2c_frozen_label,universe_b_label,conflict
0,17,divergence_expected,neutral,True
1,22,neutral,neutral,False
2,27,agreement_expected,agreement_expected,False
3,32,neutral,neutral,False
4,62,neutral,neutral,False
5,72,neutral,neutral,False
6,73,divergence_expected,neutral,True
7,74,divergence_expected,neutral,True
8,77,neutral,neutral,False
9,83,neutral,neutral,False


[C] PASS: Stage 2c overlap conflict positions -- observed={73, 17, 74}; expected={73, 74, 17}
[C] PASS: Stage 2c conflicts are divergence to neutral -- observed=[{'stage2c_frozen_label': 'divergence_expected', 'universe_b_label': 'neutral'}]; expected=[{'stage2c_frozen_label': 'divergence_expected', 'universe_b_label': 'neutral'}]


## Section D - Cross-artifact consistency

This section combines independently derived universes with the committed deep-dive and baseline artifacts to verify coverage, disjointness, and taxonomy partitioning.


In [7]:
overlap_set = set(STAGE2C_OVERLAP_POSITIONS)
deep_dive_set = {c['position'] for c in deep_dive_candidates['candidates']}
baseline_set = {b['position'] for b in test_retest_baselines['baselines']}
universe_a_set = set(derived_a)
universe_b_set = set(derived_b)

set_summary = pd.DataFrame([
    {'set': 'overlap_20', 'count': len(overlap_set), 'positions': sorted(overlap_set)},
    {'set': 'deep_dive_20', 'count': len(deep_dive_set), 'positions': sorted(deep_dive_set)},
    {'set': 'baselines_20', 'count': len(baseline_set), 'positions': sorted(baseline_set)},
    {'set': 'intersection_deep_overlap', 'count': len(deep_dive_set & overlap_set), 'positions': sorted(deep_dive_set & overlap_set)},
])
display(set_summary)
check_equal('D', 'Stage 2c overlap set subset of independently derived Universe A', overlap_set <= universe_a_set, True)
check_equal('D', 'deep-dive 20 and overlap 20 are disjoint', deep_dive_set & overlap_set, set())
check_equal('D', 'baseline 20 equals overlap 20', baseline_set, overlap_set)
check_equal('D', 'deep-dive 20 subset of independently derived Universe B', deep_dive_set <= universe_b_set, True)
check_equal('D', 'all deep-dive candidates are pending_backtest', {pos: by_position[pos]['lifecycle_state'] for pos in sorted(deep_dive_set)}, {pos: 'pending_backtest' for pos in sorted(deep_dive_set)})
check_equal('D', 'position 116 absent from reviewed candidate sets', 116 not in universe_a_set and 116 not in universe_b_set and 116 not in deep_dive_set and 116 not in baseline_set, True)

coverage_rows = []
for pos in sorted(universe_b_set):
    tier = None
    if pos in TIER1_POSITIONS:
        tier = 1
    elif pos in TIER2_POSITIONS:
        tier = 2
    if pos in TIER1_POSITIONS:
        expectation_tier = 'Tier1'
    elif pos in TIER2_POSITIONS:
        expectation_tier = 'Tier2'
    elif pos in deep_dive_set:
        expectation_tier = 'DeepDive'
    else:
        expectation_tier = 'AggregateOnly'
    membership_count = sum([pos in TIER1_POSITIONS, pos in TIER2_POSITIONS, pos in deep_dive_set, expectation_tier == 'AggregateOnly'])
    coverage_rows.append({
        'position': pos,
        'position_bucket_50': f'{((pos - 1) // 50) * 50 + 1:03d}-{min(((pos - 1) // 50) * 50 + 50, 200):03d}',
        'universe_a_label': derived_a.get(pos, {}).get('label'),
        'universe_b_label': derived_b[pos]['label'],
        'in_overlap_20': pos in overlap_set,
        'in_deep_dive_20': pos in deep_dive_set,
        'in_baselines_tier': tier,
        'expectation_tier': expectation_tier,
        'membership_count': membership_count,
    })
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df.head(25))
coverage_counts = coverage_df['expectation_tier'].value_counts().reindex(['Tier1', 'Tier2', 'DeepDive', 'AggregateOnly']).fillna(0).astype(int).to_dict()
coverage_bucket = coverage_df.groupby(['position_bucket_50', 'expectation_tier']).size().unstack(fill_value=0).reset_index()
display(coverage_bucket)
check_equal('D', 'coverage matrix has 199 replay-eligible rows', len(coverage_df), 199)
check_equal('D', 'coverage tier counts', coverage_counts, {'Tier1': 5, 'Tier2': 15, 'DeepDive': 20, 'AggregateOnly': 159})
check_equal('D', 'each replay-eligible position falls into exactly one tier', sorted(coverage_df['membership_count'].unique()), [1])


,set,count,positions
0,overlap_20,20,"[17, 22, 27, 32, 62, 72, 73, 74, 77, 83, 97, 1..."
1,deep_dive_20,20,"[1, 2, 5, 6, 8, 13, 68, 122, 127, 128, 129, 13..."
2,baselines_20,20,"[17, 22, 27, 32, 62, 72, 73, 74, 77, 83, 97, 1..."
3,intersection_deep_overlap,0,[]


[D] PASS: Stage 2c overlap set subset of independently derived Universe A -- observed=True; expected=True
[D] PASS: deep-dive 20 and overlap 20 are disjoint -- observed=set(); expected=set()
[D] PASS: baseline 20 equals overlap 20 -- observed={138, 143, 17, 147, 22, 152, 27, 32, 162, 62, 72, 73, 74, 77, 83, 97, 102, 107, 112, 117}; expected={138, 143, 17, 147, 22, 152, 27, 32, 162, 62, 72, 73, 74, 77, 83, 97, 102, 107, 112, 117}
[D] PASS: deep-dive 20 subset of independently derived Universe B -- observed=True; expected=True
[D] PASS: all deep-dive candidates are pending_backtest -- observed={1: 'pending_backtest', 2: 'pending_backtest', 5: 'pending_backtest', 6: 'pending_backtest', 8: 'pending_backtest', 13: 'pending_backtest', 68: 'pending_backtest', 122: 'pending_backtest', 127: 'pending_backtest', 128: 'pending_backtest', 129: 'pending_backtest', 132: 'pending_backtest', 172: 'pending_backtest', 173: 'pending_backtest', 178: 'pending_backtest', 182: 'pending_backtest', 187: 'pendin

,position,position_bucket_50,universe_a_label,universe_b_label,in_overlap_20,in_deep_dive_20,in_baselines_tier,expectation_tier,membership_count
0,1,001-050,NaN,divergence_expected,False,True,NaN,DeepDive,1
1,2,001-050,NaN,divergence_expected,False,True,NaN,DeepDive,1
2,3,001-050,NaN,divergence_expected,False,False,NaN,AggregateOnly,1
3,4,001-050,NaN,neutral,False,False,NaN,AggregateOnly,1
4,5,001-050,NaN,divergence_expected,False,True,NaN,DeepDive,1
5,6,001-050,NaN,divergence_expected,False,True,NaN,DeepDive,1
6,7,001-050,NaN,neutral,False,False,NaN,AggregateOnly,1
7,8,001-050,NaN,neutral,False,True,NaN,DeepDive,1
8,9,001-050,NaN,neutral,False,False,NaN,AggregateOnly,1
9,10,001-050,NaN,neutral,False,False,NaN,AggregateOnly,1


expectation_tier,position_bucket_50,AggregateOnly,DeepDive,Tier1,Tier2
0,001-050,40,6,1,3
1,051-100,42,1,3,4
2,101-150,37,5,1,6
3,151-200,40,8,0,2


[D] PASS: coverage matrix has 199 replay-eligible rows -- observed=199; expected=199
[D] PASS: coverage tier counts -- observed={'Tier1': 5, 'Tier2': 15, 'DeepDive': 20, 'AggregateOnly': 159}; expected={'Tier1': 5, 'Tier2': 15, 'DeepDive': 20, 'AggregateOnly': 159}
[D] PASS: each replay-eligible position falls into exactly one tier -- observed=[np.int64(1)]; expected=[1]


## Section E - Score fidelity and call-record reconstruction

This section rebuilds all Stage 2b and Stage 2c baseline score blocks directly from the 25 live call records, including source-record SHA256 values.


In [8]:
def extract_call_record(stage: str, idx: int) -> dict:
    path = REPO_ROOT / f'docs/d7_{stage}/call_{idx}_live_call_record.json'
    data = load_json(path)
    scores = data['critic_result']['d7b_llm_scores']
    return {
        'stage': stage,
        'call_index': idx,
        'path': str(path.relative_to(REPO_ROOT)),
        'position': data['candidate_position'],
        'plausibility': scores['semantic_plausibility'],
        'alignment': scores['semantic_theme_alignment'],
        'svr': scores['structural_variant_risk'],
        'reasoning_length': len(data['critic_result']['d7b_reasoning']),
        'source_record_sha256': sha256_file(path),
    }

stage2b_records = [extract_call_record('stage2b', i) for i in range(1, 6)]
stage2c_records = [extract_call_record('stage2c', i) for i in range(1, 21)]
stage2b_by_pos = {r['position']: r for r in stage2b_records}
stage2c_by_pos = {r['position']: r for r in stage2c_records}
call_record_df = pd.DataFrame(stage2b_records + stage2c_records)
display(call_record_df)
check_equal('E', 'loaded 25 live call records', len(call_record_df), 25)
check_equal('E', 'Stage 2b call positions', [r['position'] for r in stage2b_records], TIER1_POSITIONS)
check_equal('E', 'Stage 2c call positions', [r['position'] for r in stage2c_records], STAGE2C_OVERLAP_POSITIONS)

baseline_deltas = []
for baseline in test_retest_baselines['baselines']:
    pos = baseline['position']
    if pos in stage2b_by_pos:
        observed = stage2b_by_pos[pos]
        block = baseline['stage2b']
        for field in ['plausibility', 'alignment', 'svr', 'reasoning_length', 'source_record_sha256']:
            if block[field] != observed[field]:
                baseline_deltas.append({'position': pos, 'stage': 'stage2b', 'field': field, 'artifact': block[field], 'derived': observed[field]})
    else:
        if baseline['stage2b'] is not None:
            baseline_deltas.append({'position': pos, 'stage': 'stage2b', 'field': 'stage2b', 'artifact': baseline['stage2b'], 'derived': None})
    observed = stage2c_by_pos[pos]
    block = baseline['stage2c']
    for field in ['plausibility', 'alignment', 'svr', 'reasoning_length', 'source_record_sha256']:
        if block[field] != observed[field]:
            baseline_deltas.append({'position': pos, 'stage': 'stage2c', 'field': field, 'artifact': block[field], 'derived': observed[field]})

delta_df = pd.DataFrame(baseline_deltas, columns=['position', 'stage', 'field', 'artifact', 'derived'])
display(delta_df)
check_equal('E', 'test_retest_baselines score and SHA deltas', len(delta_df), 0)

stage2b_reasoning_lengths = [r['reasoning_length'] for r in stage2b_records]
stage2b_svr = {r['position']: r['svr'] for r in stage2b_records}
stage2c_svr = {r['position']: r['svr'] for r in stage2c_records}
check_equal('E', 'Stage 2b signoff reasoning lengths', stage2b_reasoning_lengths, EXPECTED_STAGE2B_REASONING_LENGTHS)
check_equal('E', 'Stage 2b signoff SVR values', stage2b_svr, EXPECTED_STAGE2B_SVR)
check_equal('E', 'Stage 2c signoff section 6 SVR values', stage2c_svr, EXPECTED_STAGE2C_SVR)
check_equal('E', '2D.14 documentation typo follows call-record source of truth', stage2b_by_pos[74]['alignment'], 0.85, material=False)
print('Documentation note: implementation-plan checklist 2D.14 says pos 74 Stage 2b alignment 0.90, but the call record source of truth is 0.85.')


,stage,call_index,path,position,plausibility,alignment,svr,reasoning_length,source_record_sha256
0,stage2b,1,docs/d7_stage2b/call_1_live_call_record.json,17,0.75,0.85,0.85,1237,011489dec5d6681d47f2dfd55b87673142bbc736f4989a...
1,stage2b,2,docs/d7_stage2b/call_2_live_call_record.json,73,0.75,0.85,0.85,1116,b7f04db59f5c1037dbdec0982ff2ac939a5d08a1514626...
2,stage2b,3,docs/d7_stage2b/call_3_live_call_record.json,74,0.75,0.85,0.65,1050,065fd18eab0fb13b1992d95532826417077b0e6e120d75...
3,stage2b,4,docs/d7_stage2b/call_4_live_call_record.json,97,0.75,0.90,0.95,1106,775ea1f1dca1138972d076df6acd68876abf4f0e4757ee...
4,stage2b,5,docs/d7_stage2b/call_5_live_call_record.json,138,0.75,0.90,0.15,1164,8c599d9624ff4627c204da6e485b99364444ebada0b500...
5,stage2c,1,docs/d7_stage2c/call_1_live_call_record.json,17,0.75,0.85,0.85,1061,5cd724626d9357c16dfed46466ee0be1d9c0416efb9595...
6,stage2c,2,docs/d7_stage2c/call_2_live_call_record.json,22,0.75,0.90,0.85,1192,2ef14194782860dfa728015c51e2af3e6c21da83c8b27e...
7,stage2c,3,docs/d7_stage2c/call_3_live_call_record.json,27,0.75,0.90,0.85,1033,fe0f06c6c77d9db09c67ed65106a9ae1e4e7ebdc498753...
8,stage2c,4,docs/d7_stage2c/call_4_live_call_record.json,32,0.30,0.70,0.90,1120,ff898ed4e75ef2707a09dd51b5c2aa63e11515406a33b9...
9,stage2c,5,docs/d7_stage2c/call_5_live_call_record.json,62,0.75,0.90,0.95,1134,ab2929bcde2875e0a7b2add2adb1fd92c905bd90c8b2eb...


[E] PASS: loaded 25 live call records -- observed=25; expected=25
[E] PASS: Stage 2b call positions -- observed=[17, 73, 74, 97, 138]; expected=[17, 73, 74, 97, 138]
[E] PASS: Stage 2c call positions -- observed=[17, 22, 27, 32, 62, 72, 73, 74, 77, 83, 97, 102, 107, 112, 117, 138, 143, 147, 152, 162]; expected=[17, 22, 27, 32, 62, 72, 73, 74, 77, 83, 97, 102, 107, 112, 117, 138, 143, 147, 152, 162]


,position,stage,field,artifact,derived


[E] PASS: test_retest_baselines score and SHA deltas -- observed=0; expected=0
[E] PASS: Stage 2b signoff reasoning lengths -- observed=[1237, 1116, 1050, 1106, 1164]; expected=[1237, 1116, 1050, 1106, 1164]
[E] PASS: Stage 2b signoff SVR values -- observed={17: 0.85, 73: 0.85, 74: 0.65, 97: 0.95, 138: 0.15}; expected={17: 0.85, 73: 0.85, 74: 0.65, 97: 0.95, 138: 0.15}
[E] PASS: Stage 2c signoff section 6 SVR values -- observed={17: 0.85, 22: 0.85, 27: 0.85, 32: 0.9, 62: 0.95, 72: 0.75, 73: 0.95, 74: 0.75, 77: 0.95, 83: 0.75, 97: 0.95, 102: 0.95, 107: 0.95, 112: 0.85, 117: 0.85, 138: 0.3, 143: 0.15, 147: 0.95, 152: 0.95, 162: 0.9}; expected={17: 0.85, 22: 0.85, 27: 0.85, 32: 0.9, 62: 0.95, 72: 0.75, 73: 0.95, 74: 0.75, 77: 0.95, 83: 0.75, 97: 0.95, 102: 0.95, 107: 0.95, 112: 0.85, 117: 0.85, 138: 0.3, 143: 0.15, 147: 0.95, 152: 0.95, 162: 0.9}
[E] PASS: 2D.14 documentation typo follows call-record source of truth -- observed=0.85; expected=0.85
Documentation note: implementation-plan c

## Section F - Lock 4 stratum feasibility and selection audit

This section independently re-implements Lock 4.1 stratum predicates and audits the deep-dive selection for eligibility, feasibility, and natural fresh-9 placement.


In [9]:
# Use Universe B-derived metadata for prior occurrences and max-overlap values.
non_overlap_replay_positions = sorted(universe_b_set - overlap_set)


def has_rsi_14(pos: int) -> bool:
    return 'rsi_14' in set(derived_b[pos]['factors_used'])


def stratum_predicate(stratum_id: int, pos: int) -> bool:
    row = derived_b[pos]
    theme = row['theme']
    if pos in overlap_set:
        return False
    if stratum_id == 1:
        return theme == 'volatility_regime' and not has_rsi_14(pos)
    if stratum_id == 2:
        return theme == 'volatility_regime' and has_rsi_14(pos)
    if stratum_id == 3:
        return theme == 'mean_reversion' and ((row['factor_set_size'] == 7 and row['factor_set_prior_occurrences'] >= 1) or row['max_overlap_with_priors'] >= 5)
    if stratum_id == 4:
        return 1 <= pos <= 50
    if stratum_id == 5:
        return 163 <= pos <= 200
    if stratum_id == 6:
        return theme in {'momentum', 'volume_divergence', 'calendar_effect'} and pos != 74
    raise ValueError(stratum_id)

strata_specs = {s['stratum_id']: s for s in deep_dive_candidates['strata']}
stratum_pool_rows = []
stratum_pools = {}
for sid in range(1, 7):
    pool = sorted(pos for pos in non_overlap_replay_positions if stratum_predicate(sid, pos))
    stratum_pools[sid] = pool
    spec = strata_specs[sid]
    stratum_pool_rows.append({
        'stratum_id': sid,
        'name': spec['name'],
        'derived_pool_size': len(pool),
        'artifact_fresh_pool_size': spec['fresh_pool_size'],
        'min_count': spec['min_count'],
        'max_count': spec['max_count'],
        'selected_count': spec['selected_count'],
        'selected_positions': spec['selected_positions'],
    })
stratum_pool_df = pd.DataFrame(stratum_pool_rows)
display(stratum_pool_df)
check_equal('F', 'all stratum pools meet min count', all(row['derived_pool_size'] >= row['min_count'] for row in stratum_pool_rows), True)
expected_reported_pool_sizes = {1: 7, 2: 29, 4: 46, 5: 38}
observed_reported_pool_sizes = {sid: len(stratum_pools[sid]) for sid in expected_reported_pool_sizes}
check_equal('F', 'reported non-null fresh pool sizes match independent derivation', observed_reported_pool_sizes, expected_reported_pool_sizes)
check_equal('F', 'S3 and S6 artifact pool sizes intentionally null', {3: strata_specs[3]['fresh_pool_size'], 6: strata_specs[6]['fresh_pool_size']}, {3: None, 6: None})

membership_mismatches = []
for cand in deep_dive_candidates['candidates']:
    pos = cand['position']
    sid = cand['primary_stratum_id']
    if not stratum_predicate(sid, pos):
        membership_mismatches.append({'position': pos, 'primary_stratum_id': sid})
membership_df = pd.DataFrame(membership_mismatches, columns=['position', 'primary_stratum_id'])
display(membership_df)
check_equal('F', 'all deep-dive candidates qualify for assigned primary stratum', len(membership_df), 0)

fresh9_rows = []
for cand in sorted(deep_dive_candidates['candidates'], key=lambda c: c['position']):
    if cand['position'] in FRESH_9:
        fresh9_rows.append({
            'position': cand['position'],
            'primary_stratum_id': cand['primary_stratum_id'],
            'theme': cand['theme'],
            'qualifies_primary': stratum_predicate(cand['primary_stratum_id'], cand['position']),
            'also_fits_strata': cand['also_fits_strata'],
        })
fresh9_df = pd.DataFrame(fresh9_rows)
display(fresh9_df)
check_equal('F', 'all 9 fresh-eligible candidates included', set(fresh9_df['position']), FRESH_9)
check_equal('F', 'fresh-9 candidates qualify naturally for assigned primary strata', fresh9_df['qualifies_primary'].all(), True)

fresh7 = set(stratum_pools[1])
check_equal('F', 'independent RSI-absent volatility-regime fresh-7 set', fresh7, EXPECTED_FRESH7_RSI_ABSENT_VOL)


,stratum_id,name,derived_pool_size,artifact_fresh_pool_size,min_count,max_count,selected_count,selected_positions
0,1,RSI-absent volatility_regime,7,7.0,3,5,5,"[68, 128, 173, 188, 198]"
1,2,RSI-present volatility_regime,29,29.0,2,4,2,"[8, 13]"
2,3,MR high-recurrence / high-overlap,21,NaN,3,5,5,"[122, 127, 132, 182, 197]"
3,4,Early-position 1-50 fresh,46,46.0,2,4,2,"[1, 2]"
4,5,Late-position 163-200 fresh,38,38.0,2,4,3,"[172, 178, 187]"
5,6,"Rare-families themes (momentum, volume_diverge...",118,NaN,2,4,3,"[5, 6, 129]"


[F] PASS: all stratum pools meet min count -- observed=True; expected=True
[F] PASS: reported non-null fresh pool sizes match independent derivation -- observed={1: 7, 2: 29, 4: 46, 5: 38}; expected={1: 7, 2: 29, 4: 46, 5: 38}
[F] PASS: S3 and S6 artifact pool sizes intentionally null -- observed={3: None, 6: None}; expected={3: None, 6: None}


,position,primary_stratum_id


[F] PASS: all deep-dive candidates qualify for assigned primary stratum -- observed=0; expected=0


,position,primary_stratum_id,theme,qualifies_primary,also_fits_strata
0,122,3,mean_reversion,True,[]
1,127,3,mean_reversion,True,[]
2,128,1,volatility_regime,True,[]
3,129,6,volume_divergence,True,[]
4,132,3,mean_reversion,True,[]
5,172,5,mean_reversion,True,[3]
6,178,5,volatility_regime,True,[2]
7,182,3,mean_reversion,True,[5]
8,187,5,mean_reversion,True,[3]


[F] PASS: all 9 fresh-eligible candidates included -- observed={128, 129, 132, 172, 178, 182, 122, 187, 127}; expected={128, 129, 132, 172, 178, 182, 122, 187, 127}
[F] PASS: fresh-9 candidates qualify naturally for assigned primary strata -- observed=np.True_; expected=True
[F] PASS: independent RSI-absent volatility-regime fresh-7 set -- observed={128, 3, 68, 198, 43, 173, 188}; expected={128, 3, 68, 198, 43, 188, 173}


## Section G - Position 116 isolation

This section audits position 116 as the single deterministic skipped source and confirms it is absent from every reviewed candidate set.


In [10]:
skipped_entries = [c for c in replay_candidates['candidates'] if c.get('is_skipped_source') is True]
pos116_replay = skipped_entries[0] if skipped_entries else None
pos116_required_values = {
    'position': 116,
    'firing_order': 116,
    'is_skipped_source': True,
    'lifecycle_state': 'rejected_complexity',
    'source_valid_status': 'invalid_schema',
    'theme': None,
    'factor_set_size': None,
    'factor_set_prior_occurrences': None,
    'max_overlap_with_priors': None,
    'universe_a_label': None,
    'universe_b_label': None,
}
observed_pos116_values = {k: pos116_replay.get(k) for k in pos116_required_values} if pos116_replay else {}
pos116_df = pd.DataFrame([{'field': k, 'observed': observed_pos116_values.get(k), 'expected': v, 'match': observed_pos116_values.get(k) == v} for k, v in pos116_required_values.items()])
display(pos116_df)
check_equal('G', 'exactly one replay candidate is skipped source', [c['position'] for c in skipped_entries], [116])
check_equal('G', 'position 116 replay schema required values', observed_pos116_values, pos116_required_values)
check_equal('G', 'position 116 skip_reason populated', isinstance(pos116_replay.get('skip_reason'), str) and len(pos116_replay['skip_reason']) > 20, True)
check_equal('G', 'position 116 absent from downstream reviewed candidate sets', 116 not in deep_dive_set and 116 not in baseline_set and 116 not in universe_a_set and 116 not in universe_b_set, True)


,field,observed,expected,match
0,position,116,116,True
1,firing_order,116,116,True
2,is_skipped_source,True,True,True
3,lifecycle_state,rejected_complexity,rejected_complexity,True
4,source_valid_status,invalid_schema,invalid_schema,True
5,theme,None,None,True
6,factor_set_size,None,None,True
7,factor_set_prior_occurrences,None,None,True
8,max_overlap_with_priors,None,None,True
9,universe_a_label,None,None,True


[G] PASS: exactly one replay candidate is skipped source -- observed=[116]; expected=[116]
[G] PASS: position 116 replay schema required values -- observed={'position': 116, 'firing_order': 116, 'is_skipped_source': True, 'lifecycle_state': 'rejected_complexity', 'source_valid_status': 'invalid_schema', 'theme': None, 'factor_set_size': None, 'factor_set_prior_occurrences': None, 'max_overlap_with_priors': None, 'universe_a_label': None, 'universe_b_label': None}; expected={'position': 116, 'firing_order': 116, 'is_skipped_source': True, 'lifecycle_state': 'rejected_complexity', 'source_valid_status': 'invalid_schema', 'theme': None, 'factor_set_size': None, 'factor_set_prior_occurrences': None, 'max_overlap_with_priors': None, 'universe_a_label': None, 'universe_b_label': None}
[G] PASS: position 116 skip_reason populated -- observed=True; expected=True
[G] PASS: position 116 absent from downstream reviewed candidate sets -- observed=True; expected=True


## Section H - Findings summary

This section is the executive output. It summarizes every check by section and prints the independent-verification verdict.


In [11]:
checks_df = pd.DataFrame(checks)
summary_df = checks_df.groupby('section').agg(
    total_checks=('check', 'count'),
    passed=('passed', 'sum'),
    failed=('passed', lambda s: int((~s).sum())),
    material_failed=('material', lambda s: 0),
).reset_index()
# Recompute material failures with both columns available.
material_failures_by_section = checks_df.assign(material_failed=(~checks_df['passed']) & checks_df['material']).groupby('section')['material_failed'].sum().astype(int)
summary_df['material_failed'] = summary_df['section'].map(material_failures_by_section).fillna(0).astype(int)
summary_df['verdict'] = summary_df['failed'].map(lambda n: 'PASS' if n == 0 else 'FAIL')
summary_df['notes'] = summary_df.apply(lambda r: 'all checks passed' if r['failed'] == 0 else f"{r['failed']} failed", axis=1)
display(summary_df)

failed_df = checks_df[~checks_df['passed']].copy()
display(failed_df)

if int(summary_df['material_failed'].sum()) > 0:
    final_verdict = 'INDEPENDENT_REJECT'
elif int(summary_df['failed'].sum()) > 0:
    final_verdict = 'INDEPENDENT_FLAG'
else:
    final_verdict = 'INDEPENDENT_ACCEPT'

print(f'FINAL VERDICT: {final_verdict}')
check_equal('H', 'final independent verdict', final_verdict, 'INDEPENDENT_ACCEPT')


,section,total_checks,passed,failed,material_failed,verdict,notes
0,A,6,6,0,0,PASS,all checks passed
1,B,5,5,0,0,PASS,all checks passed
2,C,10,10,0,0,PASS,all checks passed
3,D,9,9,0,0,PASS,all checks passed
4,E,8,8,0,0,PASS,all checks passed
5,F,7,7,0,0,PASS,all checks passed
6,G,4,4,0,0,PASS,all checks passed


,section,check,passed,material,notes


FINAL VERDICT: INDEPENDENT_ACCEPT
[H] PASS: final independent verdict -- observed='INDEPENDENT_ACCEPT'; expected='INDEPENDENT_ACCEPT'
